# MIT opere pubbliche incompiute 2020 - v1

Prima lettura sul file nazionale MIT `MIMS e regioni` al `31/12/2020`.

Domande guida:
- quante opere rientrano nel perimetro pulito del candidate;
- quali stati e livelli di sviluppo ricorrono di piu;
- quali cause amministrative risultano piu frequenti;
- quanto e leggibile il segnale territoriale disponibile nel mart.


In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_colwidth = 120


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'candidates').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('dataset-incubator root non trovata')


repo_root = find_repo_root(Path.cwd())
mart_path = repo_root / 'out' / 'data' / 'mart' / 'mit_opere_incompiute_2020' / '2020' / 'mart_opere_snapshot.parquet'
con = duckdb.connect()

print(repo_root)
print(mart_path)


c:\Users\gabry\OneDrive\Desktop\dataciviclab-workspace\dataset-incubator
c:\Users\gabry\OneDrive\Desktop\dataciviclab-workspace\dataset-incubator\out\data\mart\mit_opere_incompiute_2020\2020\mart_opere_snapshot.parquet


In [2]:
overview = con.execute(
    f"""
    with t as (select * from read_parquet('{mart_path.as_posix()}'))
    select
        count(*) as opere,
        count(distinct codice_cup) as cup_distinti,
        count(*) filter (where localizzazione_codice_nuts is not null and localizzazione_codice_nuts <> '') as righe_con_nuts,
        round(avg(perc_avanzamento_lavori), 2) as avanzamento_medio,
        median(perc_avanzamento_lavori) as avanzamento_mediano
    from t
    """
).df()
overview


,opere,cup_distinti,righe_con_nuts,avanzamento_medio,avanzamento_mediano
0,405,405,236,30.65,23.27


## 1. Stati dell'opera e livello di sviluppo

Il primo taglio serve a capire se il mart e sbilanciato su pochi stati amministrativi dominanti.


In [3]:
stato_opera = con.execute(
    f"""
    with t as (select * from read_parquet('{mart_path.as_posix()}'))
    select
        stato_opera,
        count(*) as opere,
        round(100.0 * count(*) / sum(count(*)) over (), 1) as quota_pct
    from t
    group by 1
    order by opere desc
    """
).df()

livello_sviluppo = con.execute(
    f"""
    with t as (select * from read_parquet('{mart_path.as_posix()}'))
    select livello_sviluppo, count(*) as opere
    from t
    group by 1
    order by opere desc
    """
).df()

stato_opera


,stato_opera,opere,quota_pct
0,"b) i lavori di realizzazione, avviati, risultano interrotti entro il termine previsto...",211,52.10
1,"a) i lavori di realizzazione, avviati, risultano interrotti oltre il termine contrattualmente...",169,41.70
2,"c) i lavori di realizzazione, ultimati, non sono stati collaudati nel termine previsto...",25,6.20


In [4]:
livello_sviluppo


,livello_sviluppo,opere
0,Art.4 c. 2 Lett. g),276
1,Art.4 c. 2 Lett. e),86
2,Art.4 c. 2 Lett. d),14
3,Art.4 c. 2 Lett. f),11
4,Art.4 c. 2 Lett. b),11
5,Art.4 c. 2 Lett. a),7


## 2. Cause ricorrenti

Le cause sono gia disponibili come flag booleani nel mart. Qui interessa soprattutto il ranking, non ancora un'interpretazione forte.


In [5]:
cause = con.execute(
    f"""
    with t as (select * from read_parquet('{mart_path.as_posix()}'))
    select *
    from (
        select 'mancanza_fondi' as causa, sum(flag_mancanza_fondi::int) as opere from t
        union all
        select 'cause_tecniche' as causa, sum(flag_cause_tecniche::int) as opere from t
        union all
        select 'sopravvenienza_norme' as causa, sum(flag_sopravvenienza_norme::int) as opere from t
        union all
        select 'fallimento_liquidazione' as causa, sum(flag_fallimento_liquidazione::int) as opere from t
        union all
        select 'mancato_interesse' as causa, sum(flag_mancato_interesse::int) as opere from t
    )
    order by opere desc
    """
).df()

cause['quota_pct_su_opere'] = (cause['opere'] / int(overview.loc[0, 'opere']) * 100).round(1)
cause


,causa,opere,quota_pct_su_opere
0,mancanza_fondi,198.00,48.90
1,cause_tecniche,141.00,34.80
2,fallimento_liquidazione,90.00,22.20
3,sopravvenienza_norme,46.00,11.40
4,mancato_interesse,21.00,5.20


## 3. Segnale territoriale disponibile

Il candidate non e ancora joinato con dizionari geografici. Il campo NUTS e presente solo su una parte del mart, quindi qui facciamo un controllo prudente di copertura e dei codici piu frequenti.


In [6]:
territorio = con.execute(
    f"""
    with t as (
        select *
        from read_parquet('{mart_path.as_posix()}')
        where localizzazione_codice_nuts is not null and localizzazione_codice_nuts <> ''
    )
    select
        localizzazione_codice_nuts,
        count(*) as opere,
        round(avg(perc_avanzamento_lavori), 2) as avanzamento_medio
    from t
    group by 1
    order by opere desc
    limit 15
    """
).df()

territorio


,localizzazione_codice_nuts,opere,avanzamento_medio
0,ITG12,25,36.41
1,ITG13,25,36.16
2,ITG14,13,38.66
3,ITG17,10,33.13
4,ITG28,10,30.66
5,ITF61,9,41.28
6,ITG27,9,34.00
7,ITF14,8,29.74
8,ITF33,8,27.72
9,ITI44,7,10.38


## 4. Lettura minima

- il mart valido contiene 405 opere e coincide con 405 `Codice_CUP` distinti;
- gli stati amministrativi prevalenti sono le opere interrotte, mentre i casi "ultimati ma non collaudati" sono molto meno numerosi;
- tra le cause, `mancanza_fondi` e `cause_tecniche` dominano il primo ranking;
- il segnale territoriale via NUTS e solo parziale: utile per un primo orientamento, non ancora per confronti forti tra territori.
